# Step 8: Baseline Nearest-Center Assignment

This step creates a baseline policy by assigning each customer demand zone to the nearest fulfillment center without considering capacity constraints.

In [1]:
# Step 8: Create baseline nearest-center assignment

from pathlib import Path
import pandas as pd
from IPython.display import display

# Set project directories
project_dir = Path("/Users/mac/Desktop/portfolio2_logistics_optimization")
data_processed_dir = project_dir / "data" / "processed"

# Load distance-cost matrix and fulfillment center capacity table
distance_cost_matrix = pd.read_csv(data_processed_dir / "distance_cost_matrix.csv")
seller_centers = pd.read_csv(data_processed_dir / "seller_fulfillment_centers_with_capacity.csv")

print("=== Input Table Shapes ===")
print("distance_cost_matrix:", distance_cost_matrix.shape)
print("seller_centers:", seller_centers.shape)

# Select the nearest fulfillment center for each customer demand zone
baseline_assignment = (
    distance_cost_matrix
    .sort_values(["customer_zone_id", "distance_km"])
    .groupby("customer_zone_id", as_index=False)
    .first()
)

print("\n=== Baseline Assignment Shape ===")
print("baseline_assignment:", baseline_assignment.shape)

print("\n=== Sample Baseline Assignments ===")
sample_columns = [
    "customer_zone_id",
    "customer_city",
    "customer_state",
    "demand",
    "seller_center_id",
    "seller_city",
    "seller_state",
    "warehouse_capacity",
    "distance_km",
    "transportation_cost"
]

display(baseline_assignment[sample_columns].head(10))

# Calculate fulfillment center assigned demand under baseline policy
baseline_center_load = (
    baseline_assignment
    .groupby("seller_center_id", as_index=False)
    .agg(
        assigned_demand=("demand", "sum"),
        assigned_zones=("customer_zone_id", "count")
    )
)

# Add warehouse capacity
baseline_center_load = baseline_center_load.merge(
    seller_centers[
        [
            "seller_center_id",
            "seller_city",
            "seller_state",
            "warehouse_capacity"
        ]
    ],
    on="seller_center_id",
    how="right"
)

# Fill centers with no assigned demand
baseline_center_load["assigned_demand"] = baseline_center_load["assigned_demand"].fillna(0).astype(int)
baseline_center_load["assigned_zones"] = baseline_center_load["assigned_zones"].fillna(0).astype(int)

# Calculate utilization and capacity violation
baseline_center_load["utilization_rate"] = (
    baseline_center_load["assigned_demand"] / baseline_center_load["warehouse_capacity"]
)

baseline_center_load["capacity_violation"] = (
    baseline_center_load["assigned_demand"] > baseline_center_load["warehouse_capacity"]
)

baseline_center_load["excess_demand"] = (
    baseline_center_load["assigned_demand"] - baseline_center_load["warehouse_capacity"]
).clip(lower=0)

print("\n=== Baseline Fulfillment Center Load ===")
display(
    baseline_center_load[
        [
            "seller_center_id",
            "seller_city",
            "seller_state",
            "assigned_demand",
            "warehouse_capacity",
            "utilization_rate",
            "capacity_violation",
            "excess_demand",
            "assigned_zones"
        ]
    ].sort_values("assigned_demand", ascending=False)
)

# Calculate baseline metrics
baseline_total_cost = baseline_assignment["transportation_cost"].sum()
baseline_total_demand = baseline_assignment["demand"].sum()
baseline_weighted_avg_distance = (
    baseline_assignment["distance_km"] * baseline_assignment["demand"]
).sum() / baseline_total_demand

baseline_total_capacity = seller_centers["warehouse_capacity"].sum()
baseline_total_excess_demand = baseline_center_load["excess_demand"].sum()
baseline_num_violated_centers = baseline_center_load["capacity_violation"].sum()
baseline_max_utilization = baseline_center_load["utilization_rate"].max()

baseline_metrics = pd.DataFrame(
    [
        {
            "policy": "baseline_nearest_center",
            "total_demand": baseline_total_demand,
            "total_capacity": baseline_total_capacity,
            "total_transportation_cost": baseline_total_cost,
            "weighted_avg_distance_km": baseline_weighted_avg_distance,
            "num_fulfillment_centers": seller_centers.shape[0],
            "num_customer_zones": baseline_assignment.shape[0],
            "num_capacity_violated_centers": baseline_num_violated_centers,
            "total_excess_demand": baseline_total_excess_demand,
            "max_utilization_rate": baseline_max_utilization
        }
    ]
)

print("\n=== Baseline Metrics ===")
display(baseline_metrics)

# Save baseline outputs
assignment_output_path = data_processed_dir / "baseline_assignment.csv"
metrics_output_path = data_processed_dir / "baseline_metrics.csv"

baseline_assignment.to_csv(assignment_output_path, index=False)
baseline_metrics.to_csv(metrics_output_path, index=False)

print("\n=== Step 8 Final Result ===")
print("Saved baseline assignment to:")
print(assignment_output_path)
print("Saved baseline metrics to:")
print(metrics_output_path)

=== Input Table Shapes ===
distance_cost_matrix: (360, 22)
seller_centers: (12, 11)

=== Baseline Assignment Shape ===
baseline_assignment: (30, 22)

=== Sample Baseline Assignments ===


,customer_zone_id,customer_city,customer_state,demand,seller_center_id,seller_city,seller_state,warehouse_capacity,distance_km,transportation_cost
0,barueri_SP,barueri,SP,469,FC_5849_sao_paulo_SP,sao paulo,SP,4732,20.027197,9.392755e+03
1,belem_PA,belem,PA,474,FC_15025_sao_jose_do_rio_preto_SP,sao jose do rio preto,SP,4717,2158.334777,1.023051e+06
2,belo_horizonte_MG,belo horizonte,MG,3077,FC_13232_campo_limpo_paulista_SP,campo limpo paulista,SP,2779,467.967149,1.439935e+06
3,brasilia_DF,brasilia,DF,2157,FC_15025_sao_jose_do_rio_preto_SP,sao jose do rio preto,SP,4717,575.369162,1.241071e+06
4,campinas_SP,campinas,SP,1622,FC_13232_campo_limpo_paulista_SP,campo limpo paulista,SP,2779,46.946222,7.614677e+04
5,contagem_MG,contagem,MG,475,FC_13232_campo_limpo_paulista_SP,campo limpo paulista,SP,2779,460.838766,2.188984e+05
6,curitiba_PR,curitiba,PR,1723,FC_5849_sao_paulo_SP,sao paulo,SP,4732,323.937679,5.581446e+05
7,florianopolis_SC,florianopolis,SC,647,FC_4782_sao_paulo_SP,sao paulo,SP,3506,468.368723,3.030346e+05
8,fortaleza_CE,fortaleza,CE,692,FC_14840_guariba_SP,guariba,SP,2708,2219.729745,1.536053e+06
9,goiania_GO,goiania,GO,781,FC_15025_sao_jose_do_rio_preto_SP,sao jose do rio preto,SP,4717,458.368170,3.579855e+05



=== Baseline Fulfillment Center Load ===


,seller_center_id,seller_city,seller_state,assigned_demand,warehouse_capacity,utilization_rate,capacity_violation,excess_demand,assigned_zones
8,FC_4160_sao_paulo_SP,sao paulo,SP,17359,2857,6.075954,True,14502,1
6,FC_8577_itaquaquecetuba_SP,itaquaquecetuba,SP,11506,3402,3.382128,True,8104,8
9,FC_13232_campo_limpo_paulista_SP,campo limpo paulista,SP,5815,2779,2.092479,True,3036,4
1,FC_5849_sao_paulo_SP,sao paulo,SP,3709,4732,0.783812,False,0,4
2,FC_15025_sao_jose_do_rio_preto_SP,sao jose do rio preto,SP,3412,4717,0.723341,False,0,3
11,FC_14840_guariba_SP,guariba,SP,3267,2708,1.206425,True,559,4
3,FC_9015_santo_andre_SP,santo andre,SP,2717,4037,0.673025,False,0,3
5,FC_4782_sao_paulo_SP,sao paulo,SP,2214,3506,0.631489,False,0,2
10,FC_3426_sao_paulo_SP,sao paulo,SP,1292,2720,0.475000,False,0,1
0,FC_14940_ibitinga_SP,ibitinga,SP,0,17934,0.000000,False,0,0



=== Baseline Metrics ===


,policy,total_demand,total_capacity,total_transportation_cost,weighted_avg_distance_km,num_fulfillment_centers,num_customer_zones,num_capacity_violated_centers,total_excess_demand,max_utilization_rate
0,baseline_nearest_center,51291,56421,1.511536e+07,294.698175,12,30,4,26201,6.075954



=== Step 8 Final Result ===
Saved baseline assignment to:
/Users/mac/Desktop/portfolio2_logistics_optimization/data/processed/baseline_assignment.csv
Saved baseline metrics to:
/Users/mac/Desktop/portfolio2_logistics_optimization/data/processed/baseline_metrics.csv
